# Chapter 10: Building the Quantitative Engine — Market Direction Prediction with ML

**Financial AI in Practice** | Manning Publications

This notebook covers:
- Building ETF-based data pipelines for market prediction
- Engineering cross-asset features from financial time series
- Implementing time-series validation with embargo periods
- Training ML models for directional market forecasting
- Evaluating trading-focused performance metrics

## 10.3 Building the Data Asset Layer: ETF Data Pipeline

### Listing 10.1 — Downloading ETF data with yfinance

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

class ETFDataPipeline:
    def __init__(self, tickers):
        self.tickers = tickers
        self.start_date = (datetime.today() -
            timedelta(days=365*10)).strftime('%Y-%m-%d')  #A
        self.end_date = datetime.today().strftime('%Y-%m-%d')
    
    def download_etf_data(self):
        print(f"Downloading historical ETF data from {self.start_date} to {self.end_date}...")
        raw_data = yf.download(
            self.tickers,
            start=self.start_date,
            end=self.end_date,
            auto_adjust=True  #B
        )
        # Extract Adjusted Close prices and Volume data separately
        adj_close = raw_data['Close'].ffill()  #C
        volume = raw_data['Volume'].ffill()
        
        # Combine adjusted closing prices and volumes with appropriate labels
        adj_close.columns = [f'{ticker}_close' for ticker in adj_close.columns]
        volume.columns = [f'{ticker}_volume' for ticker in volume.columns]
        combined_data = pd.concat([adj_close, volume], axis=1).dropna()  #D
        print(f"Completed download. Total trading days: {combined_data.shape[0]}")
        return combined_data

# Execute the pipeline
etf_list = ['SPY', 'TLT', 'IEF', 'GLD', 'SLV', 'UUP', 'EEM', 'HYG', 'LQD',
            'USO', 'DBC', 'VNQ', 'IJR', 'MTUM', 'QUAL', 'DVY', 'XLU', 'XLF', 'PAA', 'BNDX']
pipeline = ETFDataPipeline(etf_list)
etf_data = pipeline.download_etf_data()
print(etf_data.head())

#A Sets the start date exactly ten years ago from today's date
#B Uses Yahoo Finance's auto_adjust=True parameter, ensuring all historical prices reflect splits, dividends, and other corporate actions
#C Forward-fills missing values using the modern ffill() method to maintain data continuity
#D Combines adjusted closing prices and volume into a single DataFrame

### Listing 10.2 — Essential data quality validation

In [ ]:
def validate_data_quality(data):
    """Essential quality checks for financial data"""
    print("\n=== DATA QUALITY REPORT ===")
    
    # Check for missing data
    missing_pct = data.isnull().sum().sum() / (data.shape[0] * data.shape[1])
    print(f"Missing data: {missing_pct:.2%}")
    
    if missing_pct > 0.05:  # Flag if >5% missing
        print("⚠ WARNING: High missing data percentage detected")
    
    # Check for suspicious price movements (potential data errors)
    for col in data.columns:
        if '_close' in col:
            daily_returns = data[col].pct_change()
            extreme_moves = (abs(daily_returns) > 0.20).sum()  # >20% daily moves
            if extreme_moves > 5:  # More than 5 extreme moves seems suspicious
                print(f"⚠ {col}: {extreme_moves} days with >20% moves")
    
    print("Quality check complete.\n")

# Run validation on our downloaded data
validate_data_quality(etf_data)

## 10.4 Feature Engineering: Crafting Predictive Signals

### 10.4.1 Step 1: Defining the target variable (Y-label)
### Listing 10.3 — Creating realistic target labels

In [ ]:
import numpy as np

def create_target_labels(data, threshold=0.01):
    """Create next-day directional labels with realistic thresholds"""
    returns = data['SPY_close'].pct_change().shift(-1)  #A
    
    conditions = [
        (returns > threshold),   # Bullish moves
        (returns < -threshold)   # Bearish moves
    ]
    choices = [1, -1]  # 1: Up, -1: Down
    data['target'] = np.select(conditions, choices, default=0)  #B
    
    return data.dropna()

etf_data_labeled = create_target_labels(etf_data)
print(f"Target distribution:\n{etf_data_labeled['target'].value_counts()}")

#A Shifts returns by -1 to align today's features with tomorrow's movement — preventing look-ahead bias
#B Creates three-class labels: 1 (bullish), -1 (bearish), 0 (neutral)

### 10.4.2 Step 2: Engineering quant-grade predictive features
### Listing 10.4 — Basic technical features for all ETF assets

In [ ]:
import talib as ta
import pandas as pd

def engineer_basic_features(data):
    """Create fundamental technical indicators across all ETF assets"""
    feature_data = data.copy()
    
    # Apply basic technical features to all 20 ETF tickers
    etf_tickers = ['SPY', 'TLT', 'IEF', 'GLD', 'SLV', 'UUP', 'EEM', 'HYG', 'LQD',
                   'USO', 'DBC', 'VNQ', 'IJR', 'MTUM', 'QUAL', 'DVY', 'XLU', 'XLF', 'PAA', 'BNDX']
    
    for ticker in etf_tickers:
        close = feature_data[f'{ticker}_close']
        volume = feature_data[f'{ticker}_volume']
        
        # Multi-timeframe momentum
        feature_data[f'{ticker}_momentum_5d'] = close.pct_change(5)   #A
        feature_data[f'{ticker}_momentum_20d'] = close.pct_change(20)
        
        # Momentum divergence (short vs long-term)
        feature_data[f'{ticker}_momentum_divergence'] = (
            feature_data[f'{ticker}_momentum_5d'] - feature_data[f'{ticker}_momentum_20d']
        )
        
        # Volatility regime indicators
        returns = close.pct_change()
        feature_data[f'{ticker}_vol_5d'] = returns.rolling(5).std()   #B
        feature_data[f'{ticker}_vol_20d'] = returns.rolling(20).std()
        
        # Volume-price relationship (conviction indicator)
        feature_data[f'{ticker}_volume_ma'] = volume.rolling(20).mean()
        feature_data[f'{ticker}_volume_ratio'] = volume / feature_data[f'{ticker}_volume_ma']  #D
        
        # RSI for mean reversion signals
        feature_data[f'{ticker}_RSI'] = ta.RSI(close, timeperiod=14)  #C
    
    return feature_data.dropna()

#A Multi-timeframe momentum captures both short-term reversals and longer-term trends across all assets
#B Rolling volatility detects regime changes — high volatility periods cluster together
#C RSI identifies overbought/oversold conditions for mean reversion opportunities
#D Volume ratio > 1.5 often signals breakout conviction; < 0.5 suggests low-interest consolidation

### Listing 10.5 — Advanced cross-asset relationship features

In [ ]:
def engineer_cross_asset_features(data):
    """Create sophisticated features capturing inter-asset relationships"""
    feature_data = data.copy()
    
    spy_returns = feature_data['SPY_close'].pct_change()
    tlt_returns = feature_data['TLT_close'].pct_change()
    gld_returns = feature_data['GLD_close'].pct_change()
    uup_returns = feature_data['UUP_close'].pct_change()
    eem_returns = feature_data['EEM_close'].pct_change()
    hyg_returns = feature_data['HYG_close'].pct_change()
    vnq_returns = feature_data['VNQ_close'].pct_change()
    
    feature_data['SPY_TLT_correlation'] = spy_returns.rolling(20).corr(tlt_returns)  #A
    feature_data['UUP_EEM_correlation'] = uup_returns.rolling(20).corr(eem_returns)  #B
    feature_data['SPY_GLD_correlation'] = spy_returns.rolling(20).corr(gld_returns)
    
    feature_data['HYG_TLT_spread'] = (
        feature_data['HYG_momentum_5d'] - feature_data['TLT_momentum_5d']
    )  #C
    
    feature_data['VNQ_TLT_correlation'] = vnq_returns.rolling(20).corr(tlt_returns)
    
    feature_data['equity_bond_momentum_diff'] = (
        feature_data['SPY_momentum_5d'] - feature_data['TLT_momentum_5d']
    )  #D
    
    risk_on_assets = ['SPY', 'EEM', 'HYG']
    risk_off_assets = ['TLT', 'GLD']
    
    risk_on_momentum = sum([feature_data[f'{asset}_momentum_5d'] for asset in risk_on_assets]) / len(risk_on_assets)
    risk_off_momentum = sum([feature_data[f'{asset}_momentum_5d'] for asset in risk_off_assets]) / len(risk_off_assets)
    feature_data['risk_sentiment'] = risk_on_momentum - risk_off_momentum  #E
    
    return feature_data.dropna()

# Apply both feature engineering steps
etf_features_basic = engineer_basic_features(etf_data_labeled)
etf_features_complete = engineer_cross_asset_features(etf_features_basic)
print(f"Total features engineered: {etf_features_complete.shape[1]}")

#A SPY-TLT correlation breaks down during liquidity crises — powerful regime indicator
#B Strong dollar typically pressures emerging markets — captures global risk sentiment
#C Credit spread proxy using momentum differential between high-yield and treasury ETFs
#D Relative momentum between asset classes signals rotation patterns
#E Composite risk sentiment combining multiple risk-on vs risk-off assets

## 10.5 Predictive Modeling: Selecting and Validating Robust ML Methods

### 10.5.3 Understanding the prediction problem
### Listing 10.6 — Understanding our multi-class prediction problem

In [ ]:
features = etf_features_complete.drop(columns=['target'])
target = etf_features_complete['target']

print("=== TARGET VARIABLE ANALYSIS ===")
target_dist = target.value_counts().sort_index()
total_days = len(target)

print(f"Class Distribution:")
print(f"  Down (-1): {target_dist.get(-1, 0)} days ({target_dist.get(-1, 0)/total_days:.1%})")
print(f"  Neutral (0): {target_dist.get(0, 0)} days ({target_dist.get(0, 0)/total_days:.1%})")
print(f"  Up (+1): {target_dist.get(1, 0)} days ({target_dist.get(1, 0)/total_days:.1%})")
print(f"\nTotal trading days: {total_days}")
print(f"Number of features: {features.shape[1]}")

# Check for class imbalance issue
neutral_pct = target_dist.get(0, 0) / total_days
if neutral_pct > 0.6:
    print(f"\n⚠ Warning: {neutral_pct:.1%} neutral days - need balanced evaluation metrics")

### 10.5.4 Building a robust validation scheme
### Listing 10.7 — Basic time-series splitting

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Create time-series aware splits
tscv = TimeSeriesSplit(n_splits=5)

print("=== TIME SERIES SPLIT CREATION ===")
splits = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(features)):
    train_start = features.index[train_idx[0]]
    train_end = features.index[train_idx[-1]]
    test_start = features.index[test_idx[0]]
    
    print(f"Fold {fold + 1}: Train starts {train_start.date()}, Train ends {train_end.date()}, Test starts {test_start.date()}")
    
    splits.append({
        'train_idx': train_idx,
        'test_idx': test_idx
    })

print(f"\nCreated {len(splits)} time-series folds")

### Listing 10.8 — Creating embargoed time-series splits

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def create_embargoed_splits(features, embargo_days=5):  #A
    """Create time-series splits with embargo periods"""
    tscv = TimeSeriesSplit(n_splits=5)
    splits = []
    
    print("=== TIME SERIES SPLIT WITH EMBARGO ===")
    
    for fold, (train_idx, test_idx) in enumerate(tscv.split(features)):
        # Apply embargo: remove last N days from training
        embargoed_train_idx = train_idx[:-embargo_days] if len(train_idx) > embargo_days else train_idx[:len(train_idx)//2]  #B
        
        # Ensure test data starts after embargo period
        last_train_idx = train_idx[-1]
        first_valid_test_idx = last_train_idx + 1 + embargo_days  #C
        valid_test_mask = test_idx >= first_valid_test_idx
        embargoed_test_idx = test_idx[valid_test_mask]
        
        if len(embargoed_test_idx) == 0:  #D
            print(f"Fold {fold+1}: No test data after embargo. Skipping.")
            continue
        
        train_start = features.index[embargoed_train_idx[0]]
        train_end = features.index[embargoed_train_idx[-1]]
        test_start = features.index[embargoed_test_idx[0]]
        
        print(f"Fold {fold + 1}: Train ends {train_end.date()}, {2 * embargo_days}-day gap, Test starts {test_start.date()}")
        
        splits.append({
            'train_idx': embargoed_train_idx,
            'test_idx': embargoed_test_idx
        })
    
    return splits

# Create embargoed splits
splits = create_embargoed_splits(features, embargo_days=5)  #E

#A 5-day embargo prevents overlap from multi-day features like ROC_60 and MA_40
#B Safety check ensures we don't remove too much training data from early folds
#C Test data must start after original training end + embargo period for true temporal separation
#D Skip folds where embargo leaves insufficient test data
#E These properly embargoed splits become our validation framework going forward

### 10.5.5 Training the final model with optimized parameters
### Listing 10.9 — Final model training with conservative parameters

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("=== FINAL MODEL TRAINING ===")

# Parameters selected through grid search (code omitted for space constraints)
best_params = {
    'n_estimators': 150,      #A
    'max_depth': 6,           #B
    'min_samples_leaf': 20,   #C
    'max_features': 0.2,      #D
    'min_samples_split': 30   #E
}
print(f"Selected parameters: {best_params}")

# Collect predictions across all time-series folds
all_predictions = []
all_actuals = []

for fold, split in enumerate(splits):
    X_train = features.iloc[split['train_idx']]
    X_test = features.iloc[split['test_idx']]
    y_train = target.iloc[split['train_idx']]
    y_test = target.iloc[split['test_idx']]
    
    # Create fresh model for each fold to ensure independence
    model = RandomForestClassifier(  #F
        **best_params,
        random_state=42 + fold,      #G
        class_weight='balanced',
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)      #H
    predictions = model.predict(X_test)
    
    all_predictions.extend(predictions)  #I
    all_actuals.extend(y_test)
    
    print(f"Fold {fold+1}: {len(predictions)} predictions generated")

print(f"\nTotal predictions collected: {len(all_predictions)}")

#A Moderate ensemble size balances performance with computational efficiency
#B Shallow trees prevent memorizing spurious patterns in noisy financial data
#C Conservative leaf requirements ensure predictions based on sufficient evidence
#D Limited feature sampling (20%) forces model diversity and reduces overfitting
#E Higher split thresholds prevent excessive partitioning on random noise
#F Fresh model instance for each fold ensures true independence between evaluations
#G Fold-specific random seed maintains reproducibility while allowing variation
#H Each model trained only on its fold's training data
#I Accumulate all predictions for comprehensive performance analysis

### Listing 10.10 — Trading-focused evaluation

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score

print("=== TRADING-FOCUSED EVALUATION ===")

# Calculate directional accuracy (Up/Down predictions only)
directional_accuracies = []
prediction_counts = {'up': 0, 'down': 0, 'neutral': 0}

for i in range(len(all_predictions)):
    pred, actual = all_predictions[i], all_actuals[i]
    
    # Count prediction types
    if pred == 1:
        prediction_counts['up'] += 1
    elif pred == -1:
        prediction_counts['down'] += 1
    else:
        prediction_counts['neutral'] += 1

# Focus on directional accuracy
directional_mask = [(p != 0 and a != 0) for p, a in zip(all_predictions, all_actuals)]  #A
directional_preds = [p for p, mask in zip(all_predictions, directional_mask) if mask]
directional_actuals = [a for a, mask in zip(all_actuals, directional_mask) if mask]

if len(directional_preds) > 0:
    directional_correct = sum(p == a for p, a in zip(directional_preds, directional_actuals))
    directional_accuracy = directional_correct / len(directional_preds)
else:
    directional_accuracy = 0

# Summary results
balanced_acc = balanced_accuracy_score(all_actuals, all_predictions)  #B

print(f"Balanced Accuracy: {balanced_acc:.1%}")
print(f"Directional Accuracy: {directional_accuracy:.1%}")
print(f"Prediction Distribution:")
print(f"  Up signals: {prediction_counts['up']}")
print(f"  Down signals: {prediction_counts['down']}")
print(f"  Neutral: {prediction_counts['neutral']}")

#A Evaluates accuracy only when both model predicted direction AND market actually moved
#B Balanced accuracy accounts for class imbalance across all predictions